# TEXT SUMMARIZER

### DATA & LIBRARIES IMPORT 

In [1]:
# Importing required packages and libraries
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration
import warnings

/Users/purveshbhave/Desktop/summarizer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initializing the validation and train set 
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [3]:
# Quick overview of avalidation and train dataset  
print("Train Data Head:")
print(train_data.head())

print("\nValidation Data Head:")
print(val_data.head())

Train Data Head:
         id                                           dialogue  \
0  13818513  Amanda: I baked  cookies. Do you want some?\r\...   
1  13728867  Olivia: Who are you voting for in this electio...   
2  13681000  Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...   
3  13730747  Edward: Rachel, I think I'm in ove with Bella....   
4  13728094  Sam: hey  overheard rick say something\r\nSam:...   

                                             summary  
0  Amanda baked cookies and will bring Jerry some...  
1  Olivia and Olivier are voting for liberals in ...  
2  Kim may try the pomodoro technique recommended...  
3  Edward thinks he is in love with Bella. Rachel...  
4  Sam is confused, because he overheard Rick com...  

Validation Data Head:
         id                                           dialogue  \
0  13817023  A: Hi Tom, are you busy tomorrow’s afternoon?\...   
1  13716628  Emma: I’ve just fallen in love with this adven...   
2  13829420  Jackie: Madison is pre

In [4]:
train_data["dialogue"][0]

"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [5]:
# Quick look at the dimensions of validation and training set 
print("Train Data Dimensions:")
print(train_data.shape)

print("\nValidation Data Dimensions:")
print(val_data.shape)

Train Data Dimensions:
(14732, 3)

Validation Data Dimensions:
(818, 3)


In [6]:
# random sampling -  shrinking data to manageable subsets
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [7]:
# Dimensions post ramdom sampling 
print("Train Data Dimensions:")
print(train_data.shape)

print("\nValidation Data Dimensions:")
print(val_data.shape)

Train Data Dimensions:
(4000, 3)

Validation Data Dimensions:
(500, 3)


### DATA PRE-PROCESSING 

In [8]:
import re

In [9]:
# Defining a function to clean data.
def clean_data(text):
    text = re.sub(r"\r\n", " ", text) # lines
    text = re.sub(r"\s+", " ", text) # spaces
    text = re.sub(r"<.*?>", " ", text) # html tags <p> <h1>
    text = text.strip().lower() # removing trailing space and lower casing all the words 
    return text

In [10]:
# Applying cleaning function to training data and validation data. 
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

### TOKENIZATION

In [11]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [12]:
# defined a function to tokenize inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_length=512, truncation=True)
    targets = tokenizer(data["summary"], padding="max_length", max_length=150, truncation=True)

    inputs["labels"] = targets["input_ids"] # token ids => add to input as labels
    return inputs

In [13]:
# Applying the tokenize function on train and validation dataset
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [14]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [15]:
# Convert dialogue text into input token IDs

# Token conventions:
# 1 represents End Of Sequence (EOS)
# 0 represents padding tokens

# Attention mask:
# Distinguishes real tokens (1) from padding tokens (0)

# Labels:
# Tokenized summary used as the target output for training

In [16]:
len(train_dataset[0]["input_ids"]) # 512

type(train_dataset)
type(val_dataset)

list

### WORKING WITH MODEL  

In [17]:
# NLP => generation task
model = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 9253.49it/s]


In [18]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_availanle():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device: ", device)
model.to(device)

device:  mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [19]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
    # 0 => lr default
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [21]:
# train the model
trainer.train()

/Users/purveshbhave/Desktop/summarizer/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.583782,0.381428
2,0.396259,0.359447
3,0.373687,0.353606
4,0.362294,0.350072
5,0.355178,0.348950
6,0.352217,0.348391


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.39it/s]
/Users/purveshbhave/Desktop/summarizer/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.64it/s]
/Users/purveshbhave/Desktop/summarizer/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 10.63it/s]
/Users/purveshbhave/Desktop/summarizer/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████

TrainOutput(global_step=3000, training_loss=0.9039028930664063, metrics={'train_runtime': 2313.8765, 'train_samples_per_second': 10.372, 'train_steps_per_second': 1.297, 'total_flos': 3248203235328000.0, 'train_loss': 0.9039028930664063, 'epoch': 6.0})

In [22]:
# model load => fine-tune => save the model

In [23]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.59it/s]


('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [24]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 17722.03it/s]


### Test the core logic for summarization

In [25]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )
    
    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [27]:
test_dialogue = """ 
Reporter: In today’s environmental news, climate change continues to be a pressing global issue, with rising temperatures and extreme weather events affecting communities worldwide. Recent studies show an increase in heatwaves, floods, and droughts over the past decade.

Reporter: Governments and organizations are taking steps to reduce carbon emissions by investing in renewable energy sources like solar and wind. However, transitioning away from fossil fuels remains a significant challenge for many economies.

Expert: The main driver of climate change is the accumulation of greenhouse gases in the atmosphere, primarily from human activities such as burning fossil fuels and deforestation. These gases trap heat and disrupt the Earth’s natural climate systems.

Expert: Renewable energy technologies have improved significantly, making them more efficient and cost-effective. This progress is encouraging wider adoption, but infrastructure and storage limitations still need to be addressed.

Reporter: Many countries have pledged to achieve net-zero emissions in the coming decades, aiming to limit global warming. International cooperation is seen as crucial to meeting these targets.

Expert: Another important factor is public awareness and behavioral change. Reducing energy consumption, minimizing waste, and supporting sustainable practices can collectively make a significant impact.

Reporter: There are also concerns about the impact of climate change on biodiversity, as many species struggle to adapt to rapidly changing environments.

Expert: Looking ahead, combining policy action, technological innovation, and community engagement will be key to mitigating climate change and protecting the planet for future generations.

"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  climate change continues to be a pressing global issue, with rising temperatures and extreme weather events affecting communities worldwide. report: governments and organizations are taking steps to reduce carbon emissions by investing in renewable energy sources like solar and wind.


In [28]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact.
